In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os


In [ ]:
DATA_DIR = 'dataset/' 
IMG_DIRS = [
    os.path.join(DATA_DIR, 'imgs_part1'),
    os.path.join(DATA_DIR, 'imgs_part2'),
    os.path.join(DATA_DIR, 'imgs_part3')
]

METADATA_PATH = os.path.join(DATA_DIR, 'metadata.csv')

NUM_CLIENTS = 4 
IMG_SIZE = 224
BATCH_SIZE = 32
RANDOM_STATE = 42

def find_image_path(img_id):
    for folder in IMG_DIRS:
        path = os.path.join(folder, img_id)
        if os.path.exists(path):
            return path
    return None 

Langkah Ke-1 : Memuat Metadata

In [ ]:
# Membaca file metadata
df = pd.read_csv(METADATA_PATH)

# Membuat path lengkap untuk setiap gambar
df['image_path'] = df['img_id'].apply(find_image_path)

# 4. Hapus baris di mana gambar tidak ditemukan (sangat penting!)
df.dropna(subset=['image_path'], inplace=True)

# Daftar diagnosis yang dianggap ganas
MALIGNANT_DIAGNOSES = ['MEL', 'BCC', 'SCC'] # Sesuaikan dengan label di dataset Anda

# Buat kolom label biner
df['label'] = df['diagnostic'].apply(lambda x: 1 if x in MALIGNANT_DIAGNOSES else 0)

# Setelah membuat label, Anda bisa membuang kolom diagnostik asli agar tidak membingungkan
# df = df.drop(columns=['diagnostic'])
print(f"Total data yang valid: {len(df)} gambar")
df.head()

Total data yang valid: 2298 gambar


,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed,image_path,label
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1516_1765_530.png,False,dataset/imgs_part3/PAT_1516_1765_530.png,0
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,TRUE,TRUE,FALSE,TRUE,TRUE,TRUE,PAT_46_881_939.png,True,dataset/imgs_part1/PAT_46_881_939.png,1
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1545_1867_547.png,False,dataset/imgs_part3/PAT_1545_1867_547.png,0
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1989_4061_934.png,False,dataset/imgs_part3/PAT_1989_4061_934.png,0
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,TRUE,TRUE,FALSE,FALSE,TRUE,TRUE,PAT_684_1302_588.png,True,dataset/imgs_part2/PAT_684_1302_588.png,1


**Langkah ke-2 : Membagi data menjadi set Latih dan Uji berdasarkan 'patient_id'**

In [8]:
# Dapatkan daftar pasien unik
patient_ids = df['patient_id'].unique()

# Bagi daftar pasien menjadi 80% untuk latih dan 20% untuk uji
train_patient_ids, test_patient_ids = train_test_split(
    patient_ids,
    test_size=0.2,
    random_state=RANDOM_STATE
)

# Buat dataframe terpisah untuk data latih dan data uji
train_df = df[df['patient_id'].isin(train_patient_ids)].copy()
test_df = df[df['patient_id'].isin(test_patient_ids)].copy()

print(f"Jumlah data latih: {len(train_df)}")
print(f"Jumlah data uji: {len(test_df)}")

Jumlah data latih: 1822
Jumlah data uji: 476


**Langkah ke-3 : Membuat Dataset Publik (untuk Knowledge Distillation)**  
Tujuannya untuk menyiapkan satu set data kecil yang akan digunakan oleh server untuk proses agregasi Knowledge Distillation.

In [9]:
# Ambil 10% dari data latih untuk menjadi data publik
# Gunakan 'stratify' untuk memastikan proporsi label tetap seimbang
remaining_train_df, public_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df['label']
)

print(f"Jumlah data publik: {len(public_df)}")
print(f"Sisa data latih untuk klien: {len(remaining_train_df)}")
print("\nDistribusi label di data publik:")
print(public_df['label'].value_counts(normalize=True))

Jumlah data publik: 183
Sisa data latih untuk klien: 1639

Distribusi label di data publik:
label
0    0.530055
1    0.469945
Name: proportion, dtype: float64


**Langkah ke-4 : Mendistribusikan Data ke Klien**  
Tujuannya untuk membagi sisa data latih ke dalam n "klien" virtual dengan cara yang realistis (berdasarkan pasien).

In [10]:
# Dapatkan daftar pasien unik dari sisa data latih
client_patient_ids = remaining_train_df['patient_id'].unique()
np.random.shuffle(client_patient_ids) # Acak ID pasien

# Bagi ID pasien secara merata ke N klien
patient_id_splits = np.array_split(client_patient_ids, NUM_CLIENTS)

# Buat dictionary untuk menyimpan dataset setiap klien
client_data = {}
for i in range(NUM_CLIENTS):
    # Dapatkan ID pasien untuk klien saat ini
    client_pids = patient_id_splits[i]
    # Ambil semua baris data yang sesuai dengan ID pasien tersebut
    client_df = remaining_train_df[remaining_train_df['patient_id'].isin(client_pids)]
    client_data[i] = client_df
    print(f"Klien {i}: {len(client_df)} gambar dari {len(client_pids)} pasien")

Klien 0: 420 gambar dari 260 pasien
Klien 1: 397 gambar dari 260 pasien
Klien 2: 409 gambar dari 260 pasien
Klien 3: 413 gambar dari 260 pasien


**Langkah ke-5 : Membuat fungsi Load dan Pre Processing Gambar**  
Fungsinya adalah memuat satu gambar, melakukan augmentasi dan normalisasi  
Flow detail :  
1. Membaca file gambar dari hard drive sebagai data biner mentah.  
2. Mengubah data biner mentah menjadi tensor angka yang merepresentasikan piksel gambar. channels=3 memastikan gambar dibaca sebagai RGB.  
3. Menyamakan ukuran semua gambar menjadi 224x224 piksel. Ini wajib karena model CNN memerlukan input dengan dimensi yang konsisten.  
4. Augmentasi data (seperti random_flip_left_right) hanya diterapkan pada data latih untuk "memperbanyak" variasinya. Kita tidak boleh mengubah data uji.  
5. Menormalisasi nilai piksel gambar Anda agar sesuai dengan cara EfficientNetV2S dilatih. Tanpa ini, performa model akan sangat buruk.


In [11]:
def load_and_preprocess_image(image_path, label, is_training=False):

    # Baca gambar
    img = Image.open(image_path).convert("RGB")

    # Transform untuk training
    if is_training:
        transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(90),   # approximate rot90
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
    else:
        # Transform untuk validation/testing
        transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    img = transform(img)

    return img, label

**Langkah Ke-6 : Membuat Pipa Data**  
Tujuannya untuk mengubah kumpulan data statis (daftar file di DataFrame) menjadi sebuah "pipa data" yang efisien untuk TensorFlow.  
Flow Detail :  
1. Membangun alur input data menggunakan API standar TensorFlow. Alih-alih memuat semua gambar ke memori sekaligus (yang bisa menyebabkan crash), tf.data.Dataset memuat dan memprosesnya secara streaming.  
2. Membuat "sumber" dataset dari path gambar dan label yang ada di DataFrame Anda.  
3. Menerapkan fungsi load_and_preprocess_image dari Langkah 5 ke setiap item di dataset. num_parallel_calls=tf.data.AUTOTUNE memungkinkan TensorFlow memproses beberapa gambar secara paralel untuk efisiensi.  
4. Mengacak urutan data latih di setiap epoch. Ini sangat penting untuk mencegah model "menghafal" urutan data.  
5. Mengelompokkan data menjadi batch (misalnya, 32 gambar per batch). Model dilatih per batch, bukan per gambar.  
6. Sebuah trik optimasi. Ini memungkinkan GPU untuk melatih batch saat ini, sementara CPU secara simultan menyiapkan batch berikutnya di latar belakang. Ini mengurangi waktu tunggu dan mempercepat pelatihan secara signifikan.

In [10]:
def create_dataset(dataframe, is_training=False):
    # Buat dataset dari path gambar dan label di dataframe
    dataset = tf.data.Dataset.from_tensor_slices(
        (dataframe['image_path'].values, dataframe['label'].values)
    )
    # Terapkan fungsi pra-pemrosesan
    dataset = dataset.map(lambda x, y: load_and_preprocess_image(x, y, is_training), 
                          num_parallel_calls=tf.data.AUTOTUNE)
    
    if is_training:
        dataset = dataset.shuffle(buffer_size=len(dataframe))
        
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    
    return dataset

# Contoh cara membuat dataset untuk Klien 0
client_0_dataset = create_dataset(client_data[0], is_training=True)

# Membuat dataset untuk data uji
test_dataset = create_dataset(test_df)

# Membuat dataset untuk data publik
public_dataset = create_dataset(public_df)

print("\nContoh satu batch dari dataset Klien 0:")
for images, labels in client_0_dataset.take(1):
    print("Bentuk batch gambar:", images.shape)
    print("Bentuk batch label:", labels.shape)

2025-09-06 18:23:22.809901: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)



Contoh satu batch dari dataset Klien 0:


2025-09-06 18:23:28.353106: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 27433728 exceeds 10% of free system memory.
2025-09-06 18:23:33.486174: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:452] ShuffleDatasetV3:2: Filling up shuffle buffer (this may take a while): 380 of 433


Bentuk batch gambar: (32, 224, 224, 3)
Bentuk batch label: (32,)


2025-09-06 18:23:34.676719: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:482] Shuffle buffer filled.
2025-09-06 18:23:34.706205: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
